In [5]:
import pandas as pd 

df = pd.read_csv("Source Files/raw_production.csv")

In [6]:
df.head()

,Production_ID,Production_Date,Plant,Product_ID,Product_Name,Planned_Qty,Produced_Qty,Rejected_Qty
0,PR0001,2026-04-27,Plant 2,P003,Additive X,1115,1194,45
1,PR0002,2026-06-14,Plant 3,P005,Coating C,1992,1767,94
2,PR0003,2026-04-07,Plant 2,P008,Polymer F,932,979,34
3,PR0004,2026-09-06,Plant 3,P009,Solvent G,1265,1291,56
4,PR0005,2026-03-23,Plant 1,P002,Resin B,1137,1118,39


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Production_ID    320 non-null    str  
 1   Production_Date  320 non-null    str  
 2   Plant            320 non-null    str  
 3   Product_ID       320 non-null    str  
 4   Product_Name     320 non-null    str  
 5   Planned_Qty      320 non-null    int64
 6   Produced_Qty     320 non-null    int64
 7   Rejected_Qty     320 non-null    int64
dtypes: int64(3), str(5)
memory usage: 31.3 KB


In [8]:
df.describe()


,Planned_Qty,Produced_Qty,Rejected_Qty
count,320.000000,320.000000,320.000000
mean,1349.315625,1237.559375,52.318750
std,486.504218,465.981402,37.204657
min,508.000000,-100.000000,4.000000
25%,947.500000,850.250000,24.000000
50%,1323.500000,1212.000000,45.000000
75%,1776.250000,1630.250000,74.000000
max,2192.000000,2204.000000,328.000000


In [9]:
df.isnull().sum()


Production_ID      0
Production_Date    0
Plant              0
Product_ID         0
Product_Name       0
Planned_Qty        0
Produced_Qty       0
Rejected_Qty       0
dtype: int64

In [10]:

# Convert date column
df['Production_Date'] = pd.to_datetime(df['Production_Date'], errors='coerce')

# Check duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Check negative quantities
print("Negative planned quantities:", (df['Planned_Qty'] < 0).sum())
print("Negative produced quantities:", (df['Produced_Qty'] < 0).sum())
print("Negative rejected quantities:", (df['Rejected_Qty'] < 0).sum())

# Check rejected quantity greater than produced quantity
print("Rejected > Produced:",
      (df['Rejected_Qty'] > df['Produced_Qty']).sum())

# Display suspicious records
print(df[
    (df['Produced_Qty'] < 0) |
    (df['Rejected_Qty'] > df['Produced_Qty'])
])


Duplicate rows: 0
Negative planned quantities: 0
Negative produced quantities: 1
Negative rejected quantities: 0
Rejected > Produced: 1
   Production_ID Production_Date    Plant Product_ID Product_Name  \
77        PR0078      2026-08-29  Plant 2       P008    Polymer F   

    Planned_Qty  Produced_Qty  Rejected_Qty  
77          838          -100            27  


In [13]:

# Production shortfall or excess
df['Production_Variance'] = (
    df['Produced_Qty'] - df['Planned_Qty']
)

# Achievement percentage
df['Achievement_Pct'] = (
    df['Produced_Qty'] / df['Planned_Qty'] * 100
)

# Rejection rate
df['Rejection_Rate'] = (
    df['Rejected_Qty'] / df['Produced_Qty'] * 100
)

# Estimated accepted quantity
df['Accepted_Qty'] = (
    df['Produced_Qty'] - df['Rejected_Qty']
)

df.head()


#Production variance: Positive means production exceeded the plan; negative means a shortfall.



#Achievement percentage: How much of the planned production was achieved.



#Rejection rate: Percentage of produced units rejected.



#Accepted quantity: Produced quantity minus rejected quantity.

,Production_ID,Production_Date,Plant,Product_ID,Product_Name,Planned_Qty,Produced_Qty,Rejected_Qty,Production_Variance,Achievement_Pct,Rejection_Rate,Accepted_Qty
0,PR0001,2026-04-27,Plant 2,P003,Additive X,1115,1194,45,79,107.085202,3.768844,1149
1,PR0002,2026-06-14,Plant 3,P005,Coating C,1992,1767,94,-225,88.704819,5.319751,1673
2,PR0003,2026-04-07,Plant 2,P008,Polymer F,932,979,34,47,105.042918,3.472932,945
3,PR0004,2026-09-06,Plant 3,P009,Solvent G,1265,1291,56,26,102.055336,4.337723,1235
4,PR0005,2026-03-23,Plant 1,P002,Resin B,1137,1118,39,-19,98.328936,3.488372,1079


In [14]:

plant_analysis = df.groupby('Plant').agg(
    Total_Planned=('Planned_Qty', 'sum'),
    Total_Produced=('Produced_Qty', 'sum'),
    Total_Rejected=('Rejected_Qty', 'sum'),
    Production_Runs=('Production_ID', 'count')
).reset_index()

plant_analysis['Achievement_Pct'] = (
    plant_analysis['Total_Produced'] /
    plant_analysis['Total_Planned'] * 100
)

plant_analysis['Rejection_Rate'] = (
    plant_analysis['Total_Rejected'] /
    plant_analysis['Total_Produced'] * 100
)

plant_analysis.sort_values(
    'Achievement_Pct', ascending=False
)


,Plant,Total_Planned,Total_Produced,Total_Rejected,Production_Runs,Achievement_Pct,Rejection_Rate
0,Plant 1,116879,108017,3796,86,92.417800,3.514262
2,Plant 3,164923,151492,6785,120,91.856200,4.478784
1,Plant 2,149979,136510,6161,114,91.019409,4.513222


In [15]:

df['Month'] = df['Production_Date'].dt.to_period('M')

monthly_analysis = df.groupby('Month').agg(
    Planned=('Planned_Qty', 'sum'),
    Produced=('Produced_Qty', 'sum'),
    Rejected=('Rejected_Qty', 'sum')
).reset_index()

monthly_analysis


,Month,Planned,Produced,Rejected
0,2026-01,45162,42098,1371
1,2026-02,36304,34211,1496
2,2026-03,52476,46911,1802
3,2026-04,54422,49147,2211
4,2026-05,57812,52861,2329
5,2026-06,56492,51564,2433
6,2026-07,64658,60137,2865
7,2026-08,39506,35536,1428
8,2026-09,24949,23554,807
